# Fuel Crisis & Transport Strikes in the Philippines — YouTube Comment Dataset

**CS412 Final PIT**

This notebook builds a dataset of **public YouTube comments** about the fuel crisis and transport strikes in the Philippines, and translates every non-English comment into English.

### What the notebook does

| Step | What happens | Output |
|---|---|---|
| 1 | Install the libraries | – |
| 2 | Load the YouTube API key (kept secret) | – |
| 3 | Choose the videos | – |
| 4 | Check how many comments each video has | table |
| 5 | Download the comments with the YouTube Data API | `youtube_comments.csv` |
| 6 | Clean the data (empty, duplicate and spam comments) | – |
| 7 | Detect each comment's language | `language` column |
| 8 | Translate non-English comments with an AI model | `text_en` column |
| 9 | Save the final dataset | `youtube_comments_en.csv` |
| 10 | Check translation quality | samples |
| 11 | Download the files | – |

### How to run it
1. *(Recommended)* **Runtime → Change runtime type → T4 GPU**. Translation then takes about 1 minute instead of about 10.
2. Add your API key as described in **Step 2**.
3. **Runtime → Run all**, or run each cell from top to bottom with **Shift + Enter**.

## Step 1 — Install the libraries

| Library | Used for |
|---|---|
| `google-api-python-client` | Talking to the official **YouTube Data API v3** |
| `pandas` | Working with the comments as a table and saving CSV files |
| `lingua-language-detector` | Detecting the language of each comment (works offline) |
| `transformers`, `torch`, `sentencepiece` | Running Meta's **NLLB-200** translation model |

Colab already has `pandas`, `torch` and `transformers`, so only the missing ones are installed.

In [ ]:
!pip install -q google-api-python-client lingua-language-detector sentencepiece

In [ ]:
import json
import re
import time

import pandas as pd
import torch
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from lingua import Language, LanguageDetectorBuilder
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

pd.set_option("display.max_colwidth", 120)
print("Libraries loaded. GPU available:", torch.cuda.is_available())

## Step 2 — Load the YouTube API key

The YouTube Data API needs an **API key** from Google Cloud Console (*APIs & Services → Credentials*, with **YouTube Data API v3** enabled).

The key is **never typed into the notebook**, so the notebook can be shared or submitted without exposing it. It is stored in Colab's **Secrets** instead:

1. Click the **🔑 key icon** in the left sidebar.
2. Click **Add new secret**.
3. Name it `YOUTUBE_API_KEY` and paste your key as the value.
4. Turn on **Notebook access**.

If no secret is found, the cell asks you to paste the key into a hidden input box instead.

In [ ]:
try:
    from google.colab import userdata
    API_KEY = userdata.get("YOUTUBE_API_KEY")
except Exception:
    from getpass import getpass
    API_KEY = getpass("Paste your YouTube API key (hidden): ")

youtube = build("youtube", "v3", developerKey=API_KEY)
print("API key loaded:", "yes" if API_KEY else "NO - see the instructions above")

## Step 3 — Choose the videos and settings

These are the videos the comments are collected from. All of them are news reports or discussions about the **fuel crisis** and **transport strikes** in the Philippines.

* Full links, `youtu.be` links, Shorts links or bare 11-character video IDs all work.
* `MAX_COMMENTS_PER_VIDEO` stops one very popular video from making up the whole dataset.
* `INCLUDE_REPLIES = False` collects only top-level comments, not replies to other comments.

**Quota:** the API gives about **10,000 units per day**. Each request returns up to 100 comments and costs **1 unit**, so collecting about 3,000 comments costs only about 30 units.

In [ ]:
VIDEOS = [
    "https://www.youtube.com/watch?v=T0gHkSh2ExE",
    "https://www.youtube.com/watch?v=akdVN7HTQN8",
    "https://www.youtube.com/watch?v=hHIcYvg1PlA",
    "https://www.youtube.com/watch?v=PRQ9_Hu-bXk",
    "https://www.youtube.com/watch?v=jhKxOzdd_cY",
    "https://www.youtube.com/watch?v=CRU2RoTSFVA",
    "https://www.youtube.com/watch?v=ltm23zexfHk",
    "https://www.youtube.com/watch?v=CwsGTa_QZiM",
    "https://www.youtube.com/shorts/WeQSKsZuhmI",
    "https://www.youtube.com/watch?v=VavXBDH7N0w",
]

MAX_COMMENTS_PER_VIDEO = 1000   # None = every comment
INCLUDE_REPLIES = False
MIN_TOTAL_COMMENTS = 1000       # the project's minimum dataset size


def extract_video_id(value):
    """Accept a full YouTube URL, a youtu.be link, a Shorts link, or a bare ID."""
    patterns = [
        r"(?:v=)([A-Za-z0-9_-]{11})",
        r"youtu\.be/([A-Za-z0-9_-]{11})",
        r"shorts/([A-Za-z0-9_-]{11})",
        r"embed/([A-Za-z0-9_-]{11})",
    ]
    for pattern in patterns:
        match = re.search(pattern, value)
        if match:
            return match.group(1)
    if re.fullmatch(r"[A-Za-z0-9_-]{11}", value):
        return value
    raise ValueError(f"Could not find a video ID in: {value}")


video_ids = list(dict.fromkeys(extract_video_id(v) for v in VIDEOS))  # removes duplicates
print(f"{len(video_ids)} videos:", video_ids)

## Step 4 — Check how many comments each video has

Before downloading anything, the API is asked for each video's **title** and **comment count** (`videos.list`, which costs 1 unit). This shows whether the videos together can reach the **1,000-comment minimum**.

> The count includes replies, so the number of top-level comments actually collected will be a bit lower.

In [ ]:
response = youtube.videos().list(part="snippet,statistics", id=",".join(video_ids)).execute()

VIDEO_TITLES = {item["id"]: item["snippet"]["title"] for item in response["items"]}
overview = pd.DataFrame([
    {
        "video_id": item["id"],
        "title": item["snippet"]["title"],
        "channel": item["snippet"]["channelTitle"],
        "published": item["snippet"]["publishedAt"][:10],
        "comments": int(item["statistics"].get("commentCount", 0)),
    }
    for item in response["items"]
]).sort_values("comments", ascending=False)

display(overview)
print(f"Total comments available: {overview['comments'].sum():,}")

## Step 5 — Download the comments

**How it works:**

* `commentThreads().list` returns the comments of a video **100 at a time**. Each response includes a `nextPageToken`, which is sent back to get the next 100. This is called *pagination*.
* The loop stops when there are no more pages, or when `MAX_COMMENTS_PER_VIDEO` is reached.
* `textFormat="plainText"` returns the comment text without HTML tags.
* If a video has comments turned off, it is skipped and the others continue.

**Columns saved for each comment:** video ID and title, comment ID, author, text, likes, and when it was posted and edited.

In [ ]:
def comment_row(video_id, item, comment_type, parent_id=None):
    s = item["snippet"]
    return {
        "video_id": video_id,
        "video_title": VIDEO_TITLES.get(video_id, ""),
        "comment_id": item["id"],
        "type": comment_type,
        "parent_id": parent_id,
        "author": s.get("authorDisplayName"),
        "text": s.get("textOriginal"),
        "likes": s.get("likeCount", 0),
        "published_at": s.get("publishedAt"),
        "updated_at": s.get("updatedAt"),
    }


def fetch_replies(video_id, parent_id):
    rows, page_token = [], None
    while True:
        response = youtube.comments().list(
            part="snippet", parentId=parent_id, maxResults=100,
            pageToken=page_token, textFormat="plainText",
        ).execute()
        rows += [comment_row(video_id, item, "reply", parent_id) for item in response.get("items", [])]
        page_token = response.get("nextPageToken")
        if not page_token:
            return rows


def fetch_comments(video_id, limit):
    rows, count, page_token = [], 0, None
    while True:
        response = youtube.commentThreads().list(
            part="snippet", videoId=video_id, maxResults=100,
            pageToken=page_token, textFormat="plainText", order="time",
        ).execute()
        for thread in response.get("items", []):
            top = thread["snippet"]["topLevelComment"]
            rows.append(comment_row(video_id, top, "comment"))
            count += 1
            if INCLUDE_REPLIES and thread["snippet"].get("totalReplyCount", 0) > 0:
                rows += fetch_replies(video_id, top["id"])
            if limit is not None and count >= limit:
                return rows
        page_token = response.get("nextPageToken")
        if not page_token:
            return rows

In [ ]:
all_rows = []
for video_id in video_ids:
    try:
        rows = fetch_comments(video_id, MAX_COMMENTS_PER_VIDEO)
    except HttpError as e:
        reason = e.error_details[0].get("reason") if e.error_details else e.resp.status
        print(f"{video_id}: FAILED ({reason}) - comments may be disabled")
        continue
    print(f"{video_id}: {len(rows):5d} comments  |  {VIDEO_TITLES.get(video_id, '')[:70]}")
    all_rows += rows

raw = pd.DataFrame(all_rows).drop_duplicates(subset="comment_id").reset_index(drop=True)
raw.to_csv("youtube_comments.csv", index=False, encoding="utf-8-sig")  # utf-8-sig: Excel shows emojis correctly

print(f"\nCollected {len(raw):,} comments -> youtube_comments.csv")
print("Minimum of 1,000 reached ✅" if len(raw) >= MIN_TOTAL_COMMENTS
      else "⚠️ Under 1,000 - add more videos in Step 3")
raw.head()

## Step 6 — Clean the data

Three kinds of rows are removed:

1. **Empty comments**, which have no text.
2. **Duplicates**: the same comment ID collected twice.
3. **Spam**: comments longer than **3,000 characters**. In this dataset these are one account pasting the same roughly 7,000-character essay four times. They aren't real reactions to the videos, and they would take very long to translate.

`youtube_comments.csv` keeps the raw, uncleaned data. Only the final dataset is cleaned.

In [ ]:
MAX_COMMENT_CHARS = 3000

df = raw.copy()
df["text"] = df["text"].fillna("").astype(str).str.strip()

empty = df["text"] == ""
spam = df["text"].str.len() > MAX_COMMENT_CHARS
print(f"Empty comments removed: {empty.sum()}")
print(f"Spam comments removed (over {MAX_COMMENT_CHARS} characters): {spam.sum()}")
if spam.any():
    display(df.loc[spam, ["author", "video_id"]].assign(length=df.loc[spam, "text"].str.len()))

df = df[~empty & ~spam].drop_duplicates(subset="comment_id").reset_index(drop=True)
print(f"\nComments after cleaning: {len(df):,}")

## Step 7 — Detect each comment's language

Comments on Philippine videos come in **English, Tagalog, Taglish** (a mix of Tagalog and English), **Cebuano**, and sometimes other languages such as Chinese, Japanese or French.

Off-the-shelf language detectors are unreliable on **short comments** and on **Taglish**. During testing, "Insane" and "F USA!" were detected as Tagalog. So the detector is combined with a few simple rules:

| Rule | Example | Result |
|---|---|---|
| Comment has no letters | `😂😂😂` | `none`, kept as is |
| Non-Latin script | `全世界应该…` | detected language (Chinese, Japanese, Korean, Russian, …) |
| Contains a **Cebuano** word (`dili`, `naa`, `kaayo`, …) | | `ceb` |
| Contains a common **Tagalog** word (`ang`, `ng`, `mga`, `yung`, `po`, …) | `Naka focus sa impeachment` | `tl` (Taglish counts as Tagalog) |
| Detector is at least 90% sure of another language **and** there are 4+ words | `Ils devraient manifester…` | that language |
| Anything else | `Insane`, `Godbless Philippines` | `en`, kept as is |

The Tagalog word list leaves out words that are also ordinary English words, such as `at`, `may` and `man`. It also leaves out `Pinoy` and `Pilipinas`, which often appear in English sentences.

In [ ]:
# Common Filipino words. A comment containing any of these is treated as Tagalog/Taglish.
TAGALOG_WORDS = set("""
ang ng mga sa na ko mo ka ako siya hindi di yung lang naman po talaga kasi dito natin tayo
kayo sila nila wala walang ba pa rin nga si ni ito yan ano bakit paano sana kaya pero dahil
kung para pag kami namin nyo niyo ninyo akin atin lahat ganito ganyan mahal presyo gasolina
pasada tigil sobra grabe sobrang bayad pera gobyerno nasa ay ung
naku kawawa tapos dapat pwede puwede gusto alam tama mali sige oo nalang nlng lng nmn
""".split())
# Cebuano-only words -> translated as Cebuano instead of Tagalog.
CEBUANO_WORDS = set("dili naa kaayo unsa ngano gyud jud bitaw akong imong atong nimo nako diay usab karon".split())

# Languages the detector chooses between: (short code, NLLB model code)
LANGUAGES = {
    Language.ENGLISH: ("en", "eng_Latn"),
    Language.TAGALOG: ("tl", "tgl_Latn"),
    Language.SPANISH: ("es", "spa_Latn"),
    Language.INDONESIAN: ("id", "ind_Latn"),
    Language.MALAY: ("ms", "zsm_Latn"),
    Language.ITALIAN: ("it", "ita_Latn"),
    Language.FRENCH: ("fr", "fra_Latn"),
    Language.GERMAN: ("de", "deu_Latn"),
    Language.PORTUGUESE: ("pt", "por_Latn"),
    Language.VIETNAMESE: ("vi", "vie_Latn"),
    Language.RUSSIAN: ("ru", "rus_Cyrl"),
    Language.THAI: ("th", "tha_Thai"),
    Language.JAPANESE: ("ja", "jpn_Jpan"),
    Language.KOREAN: ("ko", "kor_Hang"),
    Language.CHINESE: ("zh", "zho_Hans"),
    Language.ARABIC: ("ar", "arb_Arab"),
    Language.HINDI: ("hi", "hin_Deva"),
}
CEBUANO = ("ceb", "ceb_Latn")
NO_TEXT = ("none", None)

detector = LanguageDetectorBuilder.from_languages(*LANGUAGES).build()


def detect(text):
    """Return (short_code, nllb_code) for one comment."""
    if not re.search(r"[^\W\d_]", text):                 # no letters at all (emoji only, numbers)
        return NO_TEXT
    top = detector.compute_language_confidence_values(text)[0]
    if not re.search(r"[A-Za-z]", text):                  # non-Latin script: detector is reliable
        return LANGUAGES[top.language]

    words = set(re.findall(r"[^\W\d_]+", text.lower()))
    if words & CEBUANO_WORDS:
        return CEBUANO
    if words & TAGALOG_WORDS:
        return LANGUAGES[Language.TAGALOG]
    if top.language != Language.ENGLISH and top.value >= 0.9 and len(words) >= 4:
        return LANGUAGES[top.language]
    return LANGUAGES[Language.ENGLISH]


detected = df["text"].map(detect)
df["language"] = detected.str[0]
df["nllb_code"] = detected.str[1]

print(df["language"].value_counts().to_string())

In [ ]:
# A few examples per language, to check the detection makes sense
for lang in ["tl", "en"]:
    print(f"--- {lang} ---")
    for t in df.loc[df["language"] == lang, "text"].sample(5, random_state=1):
        print("  ", t[:110].replace("\n", " "))
print("--- other languages ---")
for _, r in df[~df["language"].isin(["en", "tl", "none"])].iterrows():
    print(f"   [{r.language}] {r.text[:100]}".replace("\n", " "))

## Step 8 — Translate the non-English comments

**Model:** [`facebook/nllb-200-distilled-600M`](https://huggingface.co/facebook/nllb-200-distilled-600M) is Meta's *No Language Left Behind* model. It translates between 200 languages, including **Tagalog and Cebuano**.

**Why an offline model:** the free Google Translate web service blocked the requests (*HTTP 429 – Too Many Requests*) after about 100 comments. The NLLB model runs inside this notebook, so it has no rate limits, costs nothing and needs no account.

**How the translation is done:**
1. The model needs the **source language**, so each comment uses the code found in Step 7 (`tgl_Latn` = Tagalog, `ceb_Latn` = Cebuano, …).
2. Each comment is **split into sentences** before translating. On long inputs the model tends to skip parts or repeat itself, and sentence-by-sentence translation fixes most of this.
3. **ALL-CAPS** comments are lowercased first, because shouting confuses the model.
4. Emoji-only fragments aren't sent to the model, because it would invent text for them.
5. English comments are **copied unchanged**, not translated.

The first run downloads the model (about 2.5 GB, roughly 1 minute in Colab).

In [ ]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device).eval()
print(f"Model loaded on {device.upper()}")

In [ ]:
MAX_PIECE_CHARS = 200                          # longest piece sent to the model at once
PIECES_PER_RUN = 32 if device == "cuda" else 4  # how many pieces the model translates together


def split_pieces(text):
    """Split a comment into sentences for more faithful translation."""
    letters = [c for c in text if c.isalpha()]
    if letters and sum(c.isupper() for c in letters) / len(letters) > 0.7:
        text = text.lower()                                       # ALL-CAPS -> lowercase

    sentences = [s.strip() for s in re.split(r"(?<=[.!?…])\s+|\n+", text)]
    sentences = [" ".join(s.split()) for s in sentences if re.search(r"[^\W\d_]", s)]  # drop emoji-only bits

    pieces = []
    for s in sentences:
        while len(s) > MAX_PIECE_CHARS:                           # very long run-on sentence
            cut = max(s.rfind(",", 0, MAX_PIECE_CHARS), s.rfind(" ", 0, MAX_PIECE_CHARS))
            cut = cut if cut > 50 else MAX_PIECE_CHARS
            pieces.append(s[:cut + 1].strip())
            s = s[cut + 1:].strip()
        if pieces and len(pieces[-1]) < 20:                       # glue tiny bits ("Ayan na!") to the next
            pieces[-1] = f"{pieces[-1]} {s}"
        elif s:
            pieces.append(s)
    return pieces or [" ".join(text.split())]


def translate_pieces(pieces, src_code):
    """Translate a list of sentences from src_code to English."""
    tokenizer.src_lang = src_code
    results = []
    for i in range(0, len(pieces), PIECES_PER_RUN):
        group = pieces[i:i + PIECES_PER_RUN]
        inputs = tokenizer(group, return_tensors="pt", padding=True, truncation=True, max_length=200).to(device)
        with torch.inference_mode():
            out = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"),  # target = English
                max_new_tokens=200,
                num_beams=2,
            )
        results += tokenizer.batch_decode(out, skip_special_tokens=True)
    return results


# Example: how one comment is split
split_pieces("What happen bella? The administration failed to prepare for this type of crisis. "
             "Naka focus sa impeachment. Mga hung hang!")

In [ ]:
needs_translation = ~df["language"].isin(["en", "none"])
todo = df.loc[needs_translation, ["text", "nllb_code"]].drop_duplicates("text")
print(f"Translating {len(todo)} comments...")

translations = {}
start = time.time()
for src_code, group in todo.groupby("nllb_code"):
    texts = group["text"].tolist()
    pieces, owners = [], []
    for t in texts:
        for p in split_pieces(t):
            pieces.append(p)
            owners.append(t)
    english = translate_pieces(pieces, src_code)
    for t in texts:
        translations[t] = " ".join(e for o, e in zip(owners, english) if o == t)
    print(f"  {src_code}: {len(texts)} comments done  ({time.time() - start:.0f}s)")

print(f"Finished in {(time.time() - start) / 60:.1f} minutes")

## Step 9 — Build and save the final dataset

The final file `youtube_comments_en.csv` has all the original columns plus:

| Column | Meaning |
|---|---|
| `language` | detected language: `en`, `tl` (Tagalog/Taglish), `ceb`, `zh`, `ja`, … or `none` (emoji only) |
| `text_en` | the comment in English. English comments are copied unchanged |
| `translated` | `True` if the comment was machine-translated |

In [ ]:
df["text_en"] = df["text"].map(translations).fillna(df["text"])
df["translated"] = needs_translation
final = df.drop(columns="nllb_code")

final.to_csv("youtube_comments_en.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(final):,} comments -> youtube_comments_en.csv\n")
summary = final["language"].value_counts().rename("comments").to_frame()
summary["share"] = (summary["comments"] / len(final) * 100).round(1).astype(str) + "%"
display(summary)
print(f"Translated: {final['translated'].sum()}   Kept as is: {(~final['translated']).sum()}")

## Step 10 — Check translation quality

Machine translation isn't perfect, so check a sample by eye.

**Known limitations, worth mentioning in the report:**
* **Slang and Taglish** are sometimes mistranslated. For example, *"humihithit pa nang shabu"* (still smoking meth) came out as *"still coughing"*.
* A few translations **leave out part** of a long comment or **repeat a phrase**. The cell below flags translations that are much shorter than the original so they can be reviewed.
* Comments with no Tagalog word that the detector wasn't sure about stay **untranslated**, as English. This leaves a word untranslated rather than inventing a wrong translation.

Before quoting a translated comment in the paper, read the original `text` as well.

In [ ]:
sample = final[final["translated"]].sample(10, random_state=3)[["language", "text", "text_en"]]
display(sample)

ratio = final["text_en"].str.len() / final["text"].str.len()
suspect = final[final["translated"] & (ratio < 0.5)]
print(f"{len(suspect)} translations are less than half the original length - worth checking:")
display(suspect[["text", "text_en"]])

## Step 11 — Download the files

* `youtube_comments.csv`: the raw comments exactly as collected.
* `youtube_comments_en.csv`: the **final cleaned and translated dataset**.

Both files open in Excel or Google Sheets, with emojis and accents displayed correctly. Colab deletes its files when the session ends, so download them, or copy them to Google Drive with the second cell.

In [ ]:
from google.colab import files
files.download("youtube_comments_en.csv")
files.download("youtube_comments.csv")

In [ ]:
# Optional: save a copy to Google Drive instead
# from google.colab import drive
# drive.mount("/content/drive")
# !cp youtube_comments.csv youtube_comments_en.csv /content/drive/MyDrive/